# 01 — Data Collection: YouTube Comments via API

## Objective
Extract YouTube comments from match highlight and recap videos for the 5 target teams (Spain, Argentina, Brazil, France, England) using the YouTube Data API v3.

## Approach
- For each match in the fixture list (from football-data.org), search YouTube for highlight/recap videos using templates like `"Spain vs Germany world cup 2026 highlights"`.
- Fetch top-level comments and replies via `commentThreads` and `comments` endpoints.
- A checkpoint file stores processed video IDs so interruptions don't cause duplication.
- Track API quota usage (10,000 units/day free tier) and stop when the safety margin is reached.
- Data is saved in both Parquet (efficient, columnar) and CSV formats.

## Business value
YouTube is the largest video platform where fans discuss match outcomes in real time. Comments on highlight videos capture raw, unfiltered fan sentiment immediately after a match — ideal for measuring the emotional impact of results.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.data_collection import collect_for_matches, load_collected
from src.results_api import fetch_matches, matches_to_dataframe
from src.config import TARGET_TEAMS, YOUTUBE_DAILY_QUOTA
from src.utils import setup_logger

logger = setup_logger(__name__)

### Target teams
Defined in `src/config.py`. Adding a new team is a single-line change.

In [ ]:
print("Target teams:", TARGET_TEAMS)

### Load match fixtures
We need the fixture list to know which matches to search YouTube for.

In [ ]:
raw_matches = fetch_matches(use_cache=True)
matches_df = matches_to_dataframe(raw_matches)
print(f"Loaded {len(matches_df)} matches")
display(matches_df[['utc_date', 'home_team', 'away_team', 'status']].head())

### Run collection
For each match, searches YouTube and fetches comments. This may take time depending on the number of videos found and API rate limits.

**Quota cost**:
- Search: 100 units per query
- CommentThreads.list: 1 unit per page
- Comments.list (replies): 1 unit per page

Free tier: 10,000 units/day.

In [ ]:
# For a real run with matches, uncomment:
# df = collect_for_matches(matches_df, output_format='parquet')

# Demo: load cached data if already collected
df = load_collected(format='parquet')
print(f"Loaded {len(df)} comments")
if not df.empty:
    display(df[['text', 'teams', 'video_title', 'source']].head())

### Data overview
Check the distribution of comments by team and video.

In [ ]:
if not df.empty:
    print("\n--- Comments per team ---")
    from collections import Counter
    all_teams = []
    for teams_str in df['teams'].dropna():
        all_teams.extend([t.strip() for t in teams_str.split(',') if t.strip()])
    display(pd.Series(Counter(all_teams)).sort_values(ascending=False))

### Checkpoint status

In [ ]:
checkpoint_path = project_root / 'data' / 'raw' / '_checkpoint_youtube.json'
if checkpoint_path.exists():
    import json
    with open(checkpoint_path) as f:
        cp = json.load(f)
    print(f"Checkpoint contains {len(cp.get('processed_video_ids', []))} processed video IDs")
else:
    print("No checkpoint yet — will be created after first collection.")

---
**Next step**: [02 — Cleaning and Preprocessing](02_limpieza_preprocesamiento.ipynb)